# Repair Model Training (Hybrid ALNS)

This notebook runs the **actual** training scripts in this folder with valid arguments and stable relative paths.


In [ ]:
from pathlib import Path
import os

# Works whether notebook is launched from repo root or this directory
NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / 'train_repair_model.py').exists():
    TRAIN_DIR = NOTEBOOK_DIR
else:
    TRAIN_DIR = NOTEBOOK_DIR / '5_hybrid_ml_metaheuristics' / 'hybrid_alns' / 'training'

if not TRAIN_DIR.exists():
    raise FileNotFoundError(f'Training directory not found: {TRAIN_DIR}')

MODELS_DIR = (TRAIN_DIR.parent / 'models').resolve()
MODELS_DIR.mkdir(parents=True, exist_ok=True)

BASELINE_MODEL = MODELS_DIR / 'repair_model_baseline.pkl'
ALNS_STATES = TRAIN_DIR / 'alns_states.pkl'
FINAL_MODEL = MODELS_DIR / 'repair_model.pkl'

print('TRAIN_DIR   =', TRAIN_DIR)
print('MODELS_DIR  =', MODELS_DIR)
print('BASELINE    =', BASELINE_MODEL)
print('ALNS_STATES =', ALNS_STATES)
print('FINAL_MODEL =', FINAL_MODEL)


## 1) Train a baseline repair model
This creates a first model used to collect on-policy ALNS states.


In [ ]:
!python "{TRAIN_DIR / 'train_repair_model.py'}" \
    --instances 5000 \
    --n-min 50 \
    --n-max 200 \
    --max-negatives 5 \
    --destroy-fraction 0.20 \
    --seed 42 \
    --workers 1 \
    --output "{BASELINE_MODEL}"


## 2) Collect ALNS repair states with the baseline model
This produces `alns_states.pkl` for DAgger-lite augmentation.


In [ ]:
!python "{TRAIN_DIR / 'collect_alns_states.py'}" \
    --model-path "{BASELINE_MODEL}" \
    --instances 500 \
    --n-min 50 \
    --n-max 200 \
    --max-negatives 5 \
    --iterations 200 \
    --seed 42 \
    --output "{ALNS_STATES}"


## 3) Retrain with augmentation
This trains the final model expected by the solver path `hybrid_alns/models/repair_model.pkl`.


In [ ]:
!python "{TRAIN_DIR / 'train_repair_model.py'}" \
    --instances 5000 \
    --n-min 50 \
    --n-max 200 \
    --max-negatives 5 \
    --destroy-fraction 0.20 \
    --seed 42 \
    --workers 1 \
    --augment-with "{ALNS_STATES}" \
    --output "{FINAL_MODEL}"


In [ ]:
assert BASELINE_MODEL.exists(), f'Missing baseline model: {BASELINE_MODEL}'
assert ALNS_STATES.exists(), f'Missing ALNS states file: {ALNS_STATES}'
assert FINAL_MODEL.exists(), f'Missing final model: {FINAL_MODEL}'
print('All expected artifacts exist.')
